Import libraries

In [1]:
pip install music21

Note: you may need to restart the kernel to use updated packages.


In [2]:
import music21
from music21 import converter, stream, note, chord

c:\Users\polyx\anaconda3\envs\symb\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


In [3]:
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd
from scipy.stats import entropy as scipy_entropy 
from pathlib import Path

Open Score

In [4]:
#fetch files
openscore_dir = Path(r'C:\Users\polyx\Documents\UPF\b. 03 Analysis of Symbolic Music and Computational Musicology\Symbolic Group Project\Lieder')
output_dir    = Path(r'C:\Users\polyx\Documents\UPF\b. 03 Analysis of Symbolic Music and Computational Musicology\Symbolic Group Project\dataset')

output_dir.mkdir(parents=True, exist_ok=True)

print('OpenScore exists:', openscore_dir.exists())

OpenScore exists: True


In [5]:
#count total files in corpus
mscx_files = list(openscore_dir.rglob('*.mscx'))
print(f'Found {len(mscx_files)} .mscx files')
mscx_files[:5]  # preview

Found 1351 .mscx files


[WindowsPath('C:/Users/polyx/Documents/UPF/b. 03 Analysis of Symbolic Music and Computational Musicology/Symbolic Group Project/Lieder/scores/Zumsteeg,_Emilie/6_Lieder,_Op.4/1_Die_Kapelle/lc6159729.mscx'),
 WindowsPath('C:/Users/polyx/Documents/UPF/b. 03 Analysis of Symbolic Music and Computational Musicology/Symbolic Group Project/Lieder/scores/Zumsteeg,_Emilie/6_Lieder,_Op.4/2_Morgenfreude/lc6160298.mscx'),
 WindowsPath('C:/Users/polyx/Documents/UPF/b. 03 Analysis of Symbolic Music and Computational Musicology/Symbolic Group Project/Lieder/scores/Zumsteeg,_Emilie/6_Lieder,_Op.4/3_Religion/lc6162644.mscx'),
 WindowsPath('C:/Users/polyx/Documents/UPF/b. 03 Analysis of Symbolic Music and Computational Musicology/Symbolic Group Project/Lieder/scores/Zumsteeg,_Emilie/6_Lieder,_Op.4/4_An_meine_Zither/lc6162666.mscx'),
 WindowsPath('C:/Users/polyx/Documents/UPF/b. 03 Analysis of Symbolic Music and Computational Musicology/Symbolic Group Project/Lieder/scores/Zumsteeg,_Emilie/6_Lieder,_Op.4/

In [135]:
#count files in corpus for specific composers
for composer in ['Schubert', 'Schumann', 'Brahms']:
    count = len([f for f in mscx_files if composer.lower() in str(f).lower()])
    print(f'{composer}: {count}')

Schubert: 86
Schumann: 88
Brahms: 110


In [9]:
#choose the files(target files) which we will convert 
target_files = [f for f in mscx_files 
                if any(c.lower() in str(f).lower() 
                       for c in ['schubert', 'schumann', 'brahms'])]

print(f'Total target files: {len(target_files)}')

Total target files: 284


In [10]:
#check where musescore is 
import subprocess

# Typical MuseScore path on Windows
MSCORE = r"C:\Program Files\MuseScore 4\bin\MuseScore4.exe"

# Test if it works
result = subprocess.run([MSCORE, '--version'], capture_output=True, text=True)
print(result.stdout)

MuseScore4 4.6.5



In [11]:
#target files conversion
converted = 0
errors = []

for mscx in target_files:
    out_path = output_dir / (mscx.stem + '.mxl')
    if out_path.exists():
        continue
    result = subprocess.run(
        [MSCORE, '-o', str(out_path), str(mscx)],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        converted += 1
    else:
        errors.append(mscx.name)

print(f'Converted: {converted}')
print(f'Errors:    {len(errors)}')

Converted: 0
Errors:    0


In [12]:
#verify files conversion
mxl_files = list(output_dir.glob('*.mxl'))
print(f'MXL files found: {len(mxl_files)}')
mxl_files[:5]  # preview

MXL files found: 284


[WindowsPath('C:/Users/polyx/Documents/UPF/b. 03 Analysis of Symbolic Music and Computational Musicology/Symbolic Group Project/dataset/lc4904021.mxl'),
 WindowsPath('C:/Users/polyx/Documents/UPF/b. 03 Analysis of Symbolic Music and Computational Musicology/Symbolic Group Project/dataset/lc4919673.mxl'),
 WindowsPath('C:/Users/polyx/Documents/UPF/b. 03 Analysis of Symbolic Music and Computational Musicology/Symbolic Group Project/dataset/lc4919879.mxl'),
 WindowsPath('C:/Users/polyx/Documents/UPF/b. 03 Analysis of Symbolic Music and Computational Musicology/Symbolic Group Project/dataset/lc4926375.mxl'),
 WindowsPath('C:/Users/polyx/Documents/UPF/b. 03 Analysis of Symbolic Music and Computational Musicology/Symbolic Group Project/dataset/lc4946175.mxl')]

In [30]:
#test parse on one file
test_file = mxl_files[5]
score = converter.parse(str(test_file))
print('File:', test_file.name)
print('Parts:', [p.partName for p in score.parts])
print('Measures:', len(score.parts[0].getElementsByClass('Measure')))

File: lc4946239.mxl
Parts: ['Singstimme Voice', 'Pianoforte', 'Pianoforte']
Measures: 23


In [62]:
# create dictionary: filename --> composer
print("Scanning metadata...")

file_composer = {}
for f in mxl_files:
    try:
        s = converter.parse(str(f), forceSource=True)
        composer = s.metadata.composer if s.metadata else 'Unknown'
        file_composer[f] = composer
    except:
        file_composer[f] = 'Unknown'

# grouping per composer
schubert_files  = [f for f, c in file_composer.items() if 'Schubert' in str(c)]
brahms_files    = [f for f, c in file_composer.items() if 'Brahms' in str(c)]
schumann_files  = [f for f, c in file_composer.items() if 'Schumann' in str(c)]

print(f"Schubert: {len(schubert_files)}")
print(f"Brahms:   {len(brahms_files)}")
print(f"Schumann: {len(schumann_files)}")

Scanning metadata...


c:\Users\polyx\anaconda3\envs\symb\Lib\site-packages\music21\musicxml\xmlToM21.py:5491: MusicXMLWarning: Could not import wedge: Error in getting DynamicWedges
  warnings.warn(f'Could not import {tag}: {excep}', MusicXMLWarning)
c:\Users\polyx\anaconda3\envs\symb\Lib\site-packages\music21\musicxml\xmlToM21.py:5491: MusicXMLWarning: Could not import pedal: Error in getting PedalMark
  warnings.warn(f'Could not import {tag}: {excep}', MusicXMLWarning)
c:\Users\polyx\anaconda3\envs\symb\Lib\site-packages\music21\musicxml\xmlToM21.py:4323: MusicXMLWarning: Line <dashes> stop without start
  warnings.warn('Line <' + mxObj.tag + '> stop without start', MusicXMLWarning)


Schubert: 86
Brahms:   110
Schumann: 88


In [63]:
df_meta = pd.DataFrame([
    {'filename': f.name, 'filepath': str(f), 'composer': c}
    for f, c in file_composer.items()
])

output_dir = Path(r'C:\Users\polyx\Documents\UPF\b. 03 Analysis of Symbolic Music and Computational Musicology\Symbolic Group Project\dataset')

df_meta.to_csv(output_dir / 'metadata.csv', index=False)
print(f"Saved {len(df_meta)} entries to metadata.csv")

Saved 284 entries to metadata.csv


Compute tonal tension

Dorien Herremans, Elaine Chew. Towards emotion based music generation: A tonal tension model
based on the spiral array. The Annual Meeting of the Cognitive Science Society, Jul 2019, Montreal,
Canada. ￿hal-03277753￿

In [128]:
#define the functions
def pitch_to_spiral(midi_pitch: int) -> np.ndarray:
    r, h = 1.0, 0.1
    k = midi_pitch % 12
    angle = k * (2 * np.pi / 12) * (7/12)
    x = r * np.sin(angle)
    y = r * np.cos(angle)
    z = (midi_pitch // 12) * h
    return np.array([x, y, z])

def get_beat_unit(measure):
    ts = measure.getElementsByClass('TimeSignature')
    if not ts:
        ts = measure.getContextByClass('TimeSignature')
        if not ts: 
            return 1.0
        time_sig = ts
    else: 
        time_sig = ts[0]
    if time_sig.denominator == 8: 
        return 0.5
    else: 
        return 1.0
    
def is_anacrusis_bar(measure):
    ts = measure.getContextByClass('TimeSignature')
    if ts is None:
        return False
    expected = ts.barDuration.quarterLength
    actual = measure.duration.quarterLength
    return actual < expected

def get_bar_labels(score) -> list[str]:
    measures_p0 = list(score.parts[0].getElementsByClass('Measure'))
    labels = []
    bar = 0
    for measure in measures_p0:
        if is_anacrusis_bar(measure):
            labels.append('bar 0')
        else:
            bar += 1
            labels.append(f'bar {bar}')
    return labels
    
def tonal_tension_all_parts(score) -> list[list[float]]:
    measures_p0 = list(score.parts[0].getElementsByClass('Measure'))
    n_measures = len(measures_p0)
    
    all_measures = []
    for i in range(n_measures):
        measure = measures_p0[i]

        #if is_anacrusis_bar(measure):
        #    all_measures.append([])
        #    continue 

        beat_unit = get_beat_unit(measure)
        dur = measure.duration.quarterLength
        
        # initialization of beat slots
        beat_pitches = {}
        offset = 0.0
        while offset < dur:
            beat_pitches[round(offset, 4)] = []
            offset = round(offset + beat_unit, 4)
        
        # get pitches from all parts
        for part in score.parts:
            measures = list(part.getElementsByClass('Measure'))
            if i >= len(measures):
                continue
            for el in measures[i].flatten().notes:
                beat = round(float(el.offset) / beat_unit) * beat_unit
                beat = round(beat, 4)
                if beat not in beat_pitches:
                    continue
                if hasattr(el, 'pitches'):
                    beat_pitches[beat].extend([p.midi for p in el.pitches])
                else:
                    beat_pitches[beat].append(el.pitch.midi)
        
        # Cloud diameter per beat
        beat_tensions = []
        for beat_offset in sorted(beat_pitches.keys()):
            pitches = beat_pitches[beat_offset]
            if len(pitches) < 2:
                beat_tensions.append(0.0)
                continue
            coords = np.array([pitch_to_spiral(p) for p in pitches])
            centroid = coords.mean(axis=0)
            distances = np.linalg.norm(coords - centroid, axis=1)
            beat_tensions.append(float(distances.mean()))
        
        all_measures.append(beat_tensions)
    
    return all_measures

print('Tonal tension defined')

Tonal tension defined


In [129]:
#Test File
tt = tonal_tension_all_parts(score)
labels = get_bar_labels(score)

print(f'Measures: {len(tt)}')
for label, beats in zip(labels, tt):
    print(f'{label}: {[f"{x:.3f}" for x in beats]}')

Measures: 23
bar 1: ['0.700', '0.592', '0.705', '0.667', '0.888', '0.779']
bar 2: ['0.659', '0.626', '0.710', '0.779', '0.888', '0.668']
bar 3: ['0.659', '0.626', '0.710', '0.779', '0.810', '0.573']
bar 4: ['0.821', '0.781', '0.700', '0.641', '0.652', '0.571']
bar 5: ['0.706', '0.781', '0.897', '0.861', '0.652', '0.735']
bar 6: ['0.781', '0.702', '0.736', '0.836', '0.908', '0.881']
bar 7: ['0.871', '0.781', '0.930', '0.756', '0.716', '0.795']
bar 8: ['0.868', '0.837', '0.662', '0.686', '0.718', '0.628']
bar 9: ['0.794', '0.703', '0.393', '0.869', '0.835', '0.678']
bar 10: ['0.707', '0.050', '0.798', '0.737', '0.771', '0.512']
bar 11: ['0.700', '0.592', '0.705', '0.667', '0.888', '0.779']
bar 12: ['0.659', '0.626', '0.710', '0.779', '0.905', '0.677']
bar 13: ['0.659', '0.626', '0.710', '0.779', '0.810', '0.573']
bar 14: ['0.821', '0.781', '0.700', '0.641', '0.652', '0.571']
bar 15: ['0.706', '0.781', '0.897', '0.861', '0.652', '0.735']
bar 16: ['0.781', '0.702', '0.736', '0.836', '0.908

In [78]:
#delete this?? 
#from music21 import expressions, bar
# check for double barlines ή section markers
for i, measure in enumerate(score.parts[0].getElementsByClass('Measure')):
    rb = measure.getElementsByClass('Barline')
    for b in rb:
        if hasattr(b, 'type') and b.type in ['final', 'double']:
            print(f'Bar {i+1}: {b.type} barline')

Bar 23: final barline


Harmonic Complexity beat-level 

In [131]:
def harmonic_complexity_beat_sustained(score) -> list[list[float]]:
    measures_p0 = list(score.parts[0].getElementsByClass('Measure'))
    n_measures = len(measures_p0)
    
    all_measures = []
    for i in range(n_measures):
        measure = measures_p0[i]

        #if is_anacrusis_bar(measure):
        #    all_measures.append([])
        #    continue 

        beat_unit = get_beat_unit(measure)
        dur = measure.duration.quarterLength 
        
        # for each beat offset --> sounding pitch classes 
        beat_pcs = {}
        offset = 0.0
        while offset < dur:
            beat_pcs[round(offset, 4)] = []
            offset = round(offset + beat_unit, 4)
        
        for part in score.parts:
            measures = list(part.getElementsByClass('Measure'))
            if i >= len(measures):
                continue
            for el in measures[i].flatten().notes:
                note_start = float(el.offset)
                note_end = note_start + float(el.duration.quarterLength)
                for beat_offset in beat_pcs.keys():
                    if note_start <= beat_offset < note_end:
                        if hasattr(el, 'pitches'):
                            beat_pcs[beat_offset].extend([p.pitchClass for p in el.pitches])
                        else:
                            beat_pcs[beat_offset].append(el.pitch.pitchClass)
        
        # Entropy per beat
        beat_entropies = []
        for beat_offset in sorted(beat_pcs.keys()):
            pcs = beat_pcs[beat_offset]
            if not pcs:
                #beat_entropies.append(float(max(0.0, entropy / np.log2(12))))
                beat_entropies.append(0.0)
                continue
            pcp = np.zeros(12)
            for pc in pcs:
                pcp[pc] += 1
            pcp = pcp / pcp.sum()
            entropy = -sum(p * np.log2(p) for p in pcp if p > 0)
            beat_entropies.append(float(entropy / np.log2(12)))
        
        all_measures.append(beat_entropies)
    
    return all_measures  # list of lists

print("Harmonic Complexity Defined")

Harmonic Complexity Defined


In [117]:
hc = harmonic_complexity_beat_sustained(score)
print(f'Measures: {len(hc)}')
print()
for i, beats in enumerate(hc):
    print(f'Bar {i+1}: {[f"{x:.3f}" for x in beats]}')

Measures: 23

Bar 1: ['0.536', '0.418', '0.536', '0.418', '0.418', '0.425']
Bar 2: ['0.535', '0.425', '0.535', '0.425', '0.418', '0.418']
Bar 3: ['0.535', '0.425', '0.535', '0.425', '0.425', '0.425']
Bar 4: ['0.418', '0.226', '0.418', '0.418', '0.418', '0.425']
Bar 5: ['0.425', '0.425', '0.425', '0.425', '0.425', '0.535']
Bar 6: ['0.425', '0.418', '0.418', '0.418', '0.418', '0.418']
Bar 7: ['0.418', '0.226', '0.418', '0.418', '0.256', '0.442']
Bar 8: ['0.418', '0.418', '0.279', '0.425', '0.425', '0.425']
Bar 9: ['0.425', '0.425', '0.418', '0.425', '0.425', '0.425']
Bar 10: ['0.418', '0.418', '0.418', '0.418', '0.418', '0.256']
Bar 11: ['0.536', '0.418', '0.536', '0.418', '0.418', '0.425']
Bar 12: ['0.535', '0.425', '0.535', '0.425', '0.425', '0.425']
Bar 13: ['0.535', '0.425', '0.535', '0.425', '0.425', '0.425']
Bar 14: ['0.418', '0.226', '0.418', '0.418', '0.418', '0.425']
Bar 15: ['0.425', '0.425', '0.425', '0.425', '0.425', '0.535']
Bar 16: ['0.425', '0.418', '0.418', '0.418', '0.41

Pianistic Texture

In [118]:
def pianistic_texture_eighth(score) -> list[list[float]]:
    piano_parts = [p for p in score.parts 
                   if p.partName and any(x in p.partName.lower() 
                   for x in ['piano', 'pianoforte', 'klavier'])]
    if not piano_parts:
        piano_parts = score.parts[1:]
    
    measures_p0 = list(score.parts[0].getElementsByClass('Measure'))
    n_measures = len(measures_p0)
    
    all_measures = []
    for i in range(n_measures):
        measure = measures_p0[i]

        #if is_anacrusis_bar(measure):
        #    all_measures.append([])
        #    continue

        dur = measure.duration.quarterLength
        beat_unit = 0.5  # always eighth note
        
        # initialization of beat slots
        beat_onsets = {}
        offset = 0.0
        while offset < dur:
            beat_onsets[round(offset, 4)] = 0
            offset = round(offset + beat_unit, 4)
        
        for part in piano_parts:
            measures = list(part.getElementsByClass('Measure'))
            if i >= len(measures):
                continue
            for el in measures[i].flatten().notes:
                beat = round(float(el.offset) / beat_unit) * beat_unit
                beat = round(beat, 4)
                if beat not in beat_onsets:
                    continue
                if hasattr(el, 'pitches'):
                    beat_onsets[beat] += len(el.pitches)
                else:
                    beat_onsets[beat] += 1
        
        all_measures.append([float(beat_onsets[b]) 
                            for b in sorted(beat_onsets.keys())])
    
    return all_measures

print('Pianistic texture defined')

Pianistic texture defined


In [119]:
#test file
pt_eighth = pianistic_texture_eighth(score)

print(f'Measures: {len(pt_eighth)}')
print(f'Piano parts found: {[p.partName for p in score.parts[1:]]}')
print()
for i, beats in enumerate(pt_eighth):
    print(f'Bar {i+1}: {[round(x) for x in beats]}')

Measures: 23
Piano parts found: ['Pianoforte', 'Pianoforte']

Bar 1: [5, 1, 4, 1, 5, 1, 4, 1, 4, 1, 4, 1]
Bar 2: [5, 1, 4, 1, 5, 1, 4, 1, 4, 1, 4, 1]
Bar 3: [5, 1, 4, 1, 5, 1, 4, 1, 4, 1, 4, 1]
Bar 4: [3, 1, 3, 1, 3, 1, 4, 1, 2, 1, 4, 1]
Bar 5: [4, 1, 2, 1, 4, 1, 4, 1, 4, 1, 5, 1]
Bar 6: [4, 1, 3, 1, 2, 1, 3, 1, 3, 1, 3, 1]
Bar 7: [3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1]
Bar 8: [4, 1, 3, 1, 3, 1, 4, 1, 4, 1, 4, 1]
Bar 9: [4, 1, 4, 1, 1, 1, 4, 1, 4, 1, 4, 1]
Bar 10: [3, 1, 2, 1, 2, 1, 3, 1, 3, 1, 3, 1]
Bar 11: [5, 1, 4, 1, 5, 1, 4, 1, 4, 1, 4, 1]
Bar 12: [5, 1, 4, 1, 5, 1, 4, 1, 4, 1, 4, 1]
Bar 13: [5, 1, 4, 1, 5, 1, 4, 1, 4, 1, 4, 1]
Bar 14: [3, 1, 3, 1, 3, 1, 4, 1, 2, 1, 4, 1]
Bar 15: [4, 1, 2, 1, 4, 1, 4, 1, 4, 1, 5, 1]
Bar 16: [4, 1, 3, 1, 2, 1, 3, 1, 3, 1, 3, 1]
Bar 17: [3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1]
Bar 18: [4, 1, 3, 1, 3, 1, 4, 1, 4, 1, 4, 1]
Bar 19: [4, 1, 4, 1, 1, 1, 4, 1, 4, 1, 4, 1]
Bar 20: [3, 1, 2, 1, 2, 1, 3, 1, 3, 1, 3, 1]
Bar 21: [5, 1, 4, 1, 5, 1, 4, 1, 4, 1, 4, 1]
Ba

Melodic contour

In [125]:
def interval_to_contour(interval: float) -> float:
    if interval > 2:
        return 2.0    # Leap up
    elif interval > 0:
        return 1.0    # Step up
    elif interval == 0:
        return 0.0    # Same/Rest
    elif interval >= -2:
        return -1.0   # Step down
    else:
        return -2.0   # Leap down
    
def melodic_contour(score) -> list[list[float]]:
    voice_part = score.parts[0]
    measures_p0 = list(voice_part.getElementsByClass('Measure'))
    n_measures = len(measures_p0)
    
    voice_notes = []
    for i in range(n_measures):
        measure = measures_p0[i]
        measure_offset = float(measure.offset)
        for el in measure.flatten().notes:
            if hasattr(el, 'pitch'):
                voice_notes.append({
                    'midi': el.pitch.midi,
                    'abs_offset': measure_offset + float(el.offset),
                    'measure': i,
                    'beat_offset': float(el.offset)
                })
    
    if not voice_notes:
        return [[0.0]] * n_measures
    
    all_measures = []
    for i in range(n_measures):
        measure = measures_p0[i]
        beat_unit = get_beat_unit(measure)
        dur = measure.duration.quarterLength
        
        beat_contour = {}
        offset = 0.0
        while offset < dur:
            beat_contour[round(offset, 4)] = 0.0
            offset = round(offset + beat_unit, 4)
        
        measure_notes = [n for n in voice_notes if n['measure'] == i]
        
        for note in measure_notes:
            prev_notes = [n for n in voice_notes 
                         if n['abs_offset'] < note['abs_offset']]
            if not prev_notes:
                continue
            prev_note = prev_notes[-1]
            interval = note['midi'] - prev_note['midi']
            beat = round(float(note['beat_offset']) / beat_unit) * beat_unit
            beat = round(beat, 4)
            if beat in beat_contour:
                beat_contour[beat] = interval_to_contour(interval)
        
        all_measures.append([beat_contour[b] 
                            for b in sorted(beat_contour.keys())])
    
    return all_measures


print('Melodic contour defined')


Melodic contour defined


In [126]:
mc = melodic_contour(score)

print(f'Measures: {len(mc)}')
for i, beats in enumerate(mc):  # first 10 bars
    print(f'Bar {i+1}: {[round(x) for x in beats]}')

Measures: 23
Bar 1: [0, 0, 0, 0, 0, 0]
Bar 2: [2, -1, -1, -1, 0, 0]
Bar 3: [2, -1, -1, -1, -1, -1]
Bar 4: [2, 0, -2, 0, 0, 2]
Bar 5: [2, -1, -1, 2, -1, -1]
Bar 6: [-1, -2, 2, -1, 1, 1]
Bar 7: [0, 0, 0, 0, 0, 0]
Bar 8: [0, 0, 0, 2, -1, -1]
Bar 9: [-1, -1, 0, 1, -1, -1]
Bar 10: [-2, 0, 0, 1, 0, 0]
Bar 11: [0, 0, 0, 0, 0, 2]
Bar 12: [2, -1, -1, -1, 1, 1]
Bar 13: [1, -1, -1, -1, -1, -1]
Bar 14: [2, 0, -2, 0, 0, 2]
Bar 15: [2, -1, -1, 2, -1, -1]
Bar 16: [-1, -2, 2, -1, 1, 1]
Bar 17: [0, 0, 0, 0, 0, 0]
Bar 18: [0, 0, 0, 2, -1, -1]
Bar 19: [-1, -1, 0, 1, -1, -1]
Bar 20: [-2, 0, 0, 1, 0, 0]
Bar 21: [0, 0, 0, 0, 0, 0]
Bar 22: [0, 0, 0, 0, 0, 0]
Bar 23: [0, 0, 0, 0, 0, 0]


CSV 1: Statistically derived descriptors
CSV 2: Sequential representation

In [133]:
import warnings
warnings.filterwarnings('ignore')

def safe_entropy(flat_list):
    if not flat_list:
        return 0.0
    # Normalize to probabilities
    vals = np.array(flat_list)
    vals = vals - vals.min() + 1e-9  # shift to positive
    vals = vals / vals.sum()
    return float(scipy_entropy(vals))

# Reload metadata
df_meta = pd.read_csv(output_dir / 'metadata.csv')

# Fix composer names in metadata
df_meta['composer'] = df_meta['composer'].str.replace(
    'Johannes Brahms (1833-1897)', 'Johannes Brahms', regex=False
)

# ============================================================
# EXTRACTION LOOP
# ============================================================
records_stat = []   # for statistical CSV
records_seq  = []   # for sequential CSV

for idx, row in df_meta.iterrows():
    filepath = Path(row['filepath'])
    composer = row['composer']
    filename = row['filename']
    
    try:
        score = converter.parse(str(filepath))
    except Exception as e:
        print(f"Error parsing {filename}: {e}")
        continue
    
    # Compute features
    tt = tonal_tension_all_parts(score)
    hc = harmonic_complexity_beat_sustained(score)
    pt = pianistic_texture_eighth(score)
    mc = melodic_contour(score)
    labels = get_bar_labels(score)
    
    # ── SEQUENTIAL ──────────────────────────────────────────
    for i, label in enumerate(labels):
        records_seq.append({
            'filename': filename,
            'composer': composer,
            'bar':      label,
            'tt':       tt[i] if i < len(tt) else [],
            'hc':       hc[i] if i < len(hc) else [],
            'pt':       pt[i] if i < len(pt) else [],
            'mc':       mc[i] if i < len(mc) else [],
        })
    
    # ── STATISTICAL ─────────────────────────────────────────
    # Flatten all values to single lists
    tt_flat = [x for bar in tt for x in bar]
    hc_flat = [x for bar in hc for x in bar]
    pt_flat = [x for bar in pt for x in bar]
    mc_flat = [x for bar in mc for x in bar]
    
    records_stat.append({
        'filename': filename,
        'composer': composer,
        'tt_mean':    np.mean(tt_flat) if tt_flat else 0.0,
        'tt_std':     np.std(tt_flat)  if tt_flat else 0.0,
        'tt_entropy': safe_entropy(tt_flat),
        'hc_mean':    np.mean(hc_flat) if hc_flat else 0.0,
        'hc_std':     np.std(hc_flat)  if hc_flat else 0.0,
        'hc_entropy': safe_entropy(hc_flat),
        'pt_mean':    np.mean(pt_flat) if pt_flat else 0.0,
        'pt_std':     np.std(pt_flat)  if pt_flat else 0.0,
        'pt_entropy': safe_entropy(pt_flat),
        'mc_mean':    np.mean(mc_flat) if mc_flat else 0.0,
        'mc_std':     np.std(mc_flat)  if mc_flat else 0.0,
        'mc_entropy': safe_entropy(mc_flat),
    })  
    
    if (idx + 1) % 20 == 0:
        print(f"Processed {idx+1}/{len(df_meta)} files...")

# ── SAVE ────────────────────────────────────────────────────
df_stat = pd.DataFrame(records_stat)
df_seq  = pd.DataFrame(records_seq)

df_stat.to_csv(output_dir / 'features_statistical.csv', index=False)
df_seq.to_csv(output_dir  / 'features_sequential.csv',  index=False)

print(f"\nDone!")
print(f"Statistical CSV: {len(df_stat)} songs")
print(f"Sequential CSV:  {len(df_seq)} bars")

Processed 20/284 files...
Processed 40/284 files...
Processed 60/284 files...
Processed 80/284 files...
Processed 100/284 files...
Processed 120/284 files...
Processed 140/284 files...
Processed 160/284 files...
Processed 180/284 files...
Processed 200/284 files...
Processed 220/284 files...
Processed 240/284 files...
Processed 260/284 files...
Processed 280/284 files...

Done!
Statistical CSV: 284 songs
Sequential CSV:  15294 bars
